# Análise de Correspondência Espacial: Vetor de Eventos × Produto AQ1KM

Avalia a concordância geométrica entre **polígonos de eventos** (formato GeoJSON) e **detecções de queimada do produto AQ1KM** (formato Shapefile, resolução nominal de 1 km), circunscritas ao território brasileiro.

**Insumos**

- `eventos_YYYYMMDD_YYYYMMDD.geojson` — feições poligonais de eventos consolidados.
- `YYYY_MM_aq1km.shp` — pixels vetorizados de detecção do produto AQ1KM.
- Contorno nacional (BRA) para recorte espacial.

**Convenções metodológicas**

- Métricas de extensão expressas em **km²**.
- Cálculos de área executados em **EPSG:6933** (Equal Earth — projeção equivalente, preserva a métrica de área).
- Junção espacial com predicado `intersects` (qualquer sobreposição topológica caracteriza correspondência).

In [ ]:
# ── Dependências ──────────────────────────────────────────────────────────────
import geopandas as gpd
import pandas as pd
import leafmap
import warnings
warnings.filterwarnings('ignore')

# ── Caminhos / URLs ───────────────────────────────────────────────────────────
PATH_GEOJSON   = './eventos_20240801_20240830.geojson'
PATH_SHAPEFILE = './2024_08_aq1km.shp'
URL_BRASIL     = 'https://raw.githubusercontent.com/johan/world.geo.json/master/countries/BRA.geo.json'

# ── Constantes ────────────────────────────────────────────────────────────────
CRS_GEOGRAFICO = 4326   # WGS 84 — referência geográfica de trabalho
CRS_AREA_IGUAL = 6933   # Equal Earth — projeção equivalente para áreas
M2_POR_KM2     = 1e6    # 1 km² = 1.000.000 m²

# ── Estilos cartográficos ─────────────────────────────────────────────────────
STYLE_BRASIL      = {'stroke': True, 'color': '#ffffff', 'weight': 1, 'fill': False}
STYLE_AQ1KM_ISOL  = {'color': '#FFA500', 'fillColor': '#FFD700', 'weight': 2, 'fillOpacity': 0.8}
STYLE_EVENTO_ISOL = {'color': '#FF0303', 'fillColor': '#FF0000', 'weight': 2, 'fillOpacity': 0.8}
STYLE_AQ1KM_PAR   = {'color': '#00897B', 'fillColor': '#26A69A', 'weight': 2, 'fillOpacity': 0.7}
STYLE_EVENTO_PAR  = {'color': '#1565C0', 'fillColor': '#42A5F5', 'weight': 2, 'fillOpacity': 0.7}

# ── Funções utilitárias ───────────────────────────────────────────────────────
def area_km2(gdf: gpd.GeoDataFrame) -> float:
    """Soma das áreas em km² após reprojeção para EPSG:6933."""
    return gdf.to_crs(epsg=CRS_AREA_IGUAL).geometry.area.sum() / M2_POR_KM2


def feicoes_sem_correspondencia(gdf_alvo: gpd.GeoDataFrame,
                                gdf_referencia: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Feições de `gdf_alvo` sem intersecção topológica em `gdf_referencia`."""
    juncao = gpd.sjoin(gdf_alvo, gdf_referencia, how='left', predicate='intersects')
    return (juncao[juncao['index_right'].isna()]
            .drop(columns='index_right')
            .copy())


def criar_mapa_base(gdf_brasil: gpd.GeoDataFrame) -> leafmap.Map:
    """Mapa interativo centrado no Brasil com camada de contorno nacional."""
    m = leafmap.Map(center=[-15, -55], zoom=4)
    m.add_basemap('HYBRID')
    m.add_gdf(gdf_brasil, layer_name='Limite Brasil', style=STYLE_BRASIL)
    return m


def renderizar_mapa(gdf_brasil: gpd.GeoDataFrame, camadas: list) -> leafmap.Map:
    """Renderiza mapa com lista de camadas no formato (gdf, nome, estilo)."""
    m = criar_mapa_base(gdf_brasil)
    for gdf, nome, estilo in camadas:
        if len(gdf) > 0:
            m.add_gdf(gdf, layer_name=nome, style=estilo)
    return m

## 1. Carregamento e harmonização de CRS

Os três conjuntos são lidos, normalizados para EPSG:4326 e o produto AQ1KM é recortado pelo polígono nacional.

In [ ]:
gdf_eventos = gpd.read_file(PATH_GEOJSON).to_crs(epsg=CRS_GEOGRAFICO)
gdf_brasil  = gpd.read_file(URL_BRASIL).to_crs(epsg=CRS_GEOGRAFICO)
gdf_aq1km   = gpd.read_file(PATH_SHAPEFILE)

# Atribui CRS caso ausente; reprojeta caso divergente
if gdf_aq1km.crs is None:
    gdf_aq1km = gdf_aq1km.set_crs(epsg=CRS_GEOGRAFICO)
else:
    gdf_aq1km = gdf_aq1km.to_crs(epsg=CRS_GEOGRAFICO)

# Recorte espacial ao território nacional
gdf_aq1km_br = gpd.clip(gdf_aq1km, gdf_brasil)

## 2. Estatísticas descritivas dos insumos

Cardinalidade (nº de feições) e extensão territorial total por conjunto.

In [ ]:
n_eventos        = len(gdf_eventos)
n_aq1km          = len(gdf_aq1km_br)
area_eventos_km2 = area_km2(gdf_eventos)
area_aq1km_km2   = area_km2(gdf_aq1km_br)

display(pd.DataFrame({
    'Conjunto':        ['Eventos (GeoJSON)', 'AQ1KM no Brasil (Shapefile)'],
    'Nº de feições':   [n_eventos, n_aq1km],
    'Extensão (km²)':  [round(area_eventos_km2, 2), round(area_aq1km_km2, 2)],
}))

## 3. Identificação de feições sem correspondência espacial

Junção espacial bidirecional com predicado `intersects` separa as feições em duas classes:

- **Isoladas** — sem qualquer intersecção topológica no conjunto oposto.
- **Pareadas** — com pelo menos uma intersecção topológica no conjunto oposto.

Detecções AQ1KM isoladas indicam focos não consolidados como evento; eventos isolados indicam polígonos derivados de fontes distintas do produto AQ1KM (ou ruído).

In [ ]:
gdf_aq1km_isol   = feicoes_sem_correspondencia(gdf_aq1km_br, gdf_eventos)
gdf_eventos_isol = feicoes_sem_correspondencia(gdf_eventos, gdf_aq1km_br)

n_aq1km_isol         = len(gdf_aq1km_isol)
n_eventos_isol       = len(gdf_eventos_isol)
area_aq1km_isol_km2  = area_km2(gdf_aq1km_isol)
area_eventos_isol_km2 = area_km2(gdf_eventos_isol)

resumo = pd.DataFrame({
    'Métrica': [
        '1. Nº total de feições',
        '2. Extensão total (km²)',
        '3. Feições isoladas (nº)',
        '4. Taxa de isolamento (%)',
        '5. Extensão isolada (km²)',
        '6. Fração da extensão isolada (%)',
    ],
    'Eventos (GeoJSON)': [
        f'{n_eventos:,}',
        f'{area_eventos_km2:,.2f}',
        f'{n_eventos_isol:,}',
        f'{n_eventos_isol / n_eventos * 100:.2f}',
        f'{area_eventos_isol_km2:,.2f}',
        f'{area_eventos_isol_km2 / area_eventos_km2 * 100:.2f}',
    ],
    'AQ1KM (Shapefile)': [
        f'{n_aq1km:,}',
        f'{area_aq1km_km2:,.2f}',
        f'{n_aq1km_isol:,}',
        f'{n_aq1km_isol / n_aq1km * 100:.2f}',
        f'{area_aq1km_isol_km2:,.2f}',
        f'{area_aq1km_isol_km2 / area_aq1km_km2 * 100:.2f}',
    ],
})
display(resumo)

dif_relativa = (area_eventos_km2 - area_aq1km_km2) / area_aq1km_km2 * 100
print(f'\nDiferença relativa de extensão (Eventos × AQ1KM): {dif_relativa:+.2f}%')

## 4. Quantificação geométrica das sobreposições

Sobre o subconjunto **pareado** (feições com correspondência confirmada), executa-se três operações topológicas em EPSG:6933:

- **Intersecção** (Eventos ∩ AQ1KM) — área comum.
- **Diferença Eventos − AQ1KM** — extensão de eventos não coberta por detecções.
- **Diferença AQ1KM − Eventos** — extensão de detecções não coberta por eventos.

Adicionalmente, o **índice de Jaccard** (IoU = |A ∩ B| / |A ∪ B|) sintetiza a concordância geométrica em escala [0, 1].

In [ ]:
# Subconjuntos pareados (complemento dos isolados)
gdf_aq1km_par   = gdf_aq1km_br.drop(index=gdf_aq1km_isol.index)
gdf_eventos_par = gdf_eventos.drop(index=gdf_eventos_isol.index)

n_aq1km_par   = len(gdf_aq1km_par)
n_eventos_par = len(gdf_eventos_par)

print(f'Detecções AQ1KM pareadas : {n_aq1km_par:,} de {n_aq1km:,} ({n_aq1km_par / n_aq1km * 100:.2f}%)')
print(f'Eventos pareados         : {n_eventos_par:,} de {n_eventos:,} ({n_eventos_par / n_eventos * 100:.2f}%)')

# Reprojeção única para sistema de área igual
ev_proj = gdf_eventos_par.to_crs(epsg=CRS_AREA_IGUAL)
aq_proj = gdf_aq1km_par.to_crs(epsg=CRS_AREA_IGUAL)

# Operações topológicas
gdf_intersec   = gpd.overlay(ev_proj, aq_proj, how='intersection', keep_geom_type=True)
gdf_dif_evento = gpd.overlay(ev_proj, aq_proj, how='difference',   keep_geom_type=True)
gdf_dif_aq1km  = gpd.overlay(aq_proj, ev_proj, how='difference',   keep_geom_type=True)

# Áreas (km²) — reaproveita a soma já em metros quadrados
area_intersec_km2   = gdf_intersec.geometry.area.sum() / M2_POR_KM2
area_dif_evento_km2 = gdf_dif_evento.geometry.area.sum() / M2_POR_KM2
area_dif_aq1km_km2  = gdf_dif_aq1km.geometry.area.sum() / M2_POR_KM2
area_uniao_km2      = area_intersec_km2 + area_dif_evento_km2 + area_dif_aq1km_km2
iou                 = area_intersec_km2 / area_uniao_km2 if area_uniao_km2 > 0 else 0

display(pd.DataFrame({
    'Operação geométrica': [
        'Intersecção (Eventos ∩ AQ1KM)',
        'Diferença Eventos − AQ1KM',
        'Diferença AQ1KM − Eventos',
        'União (Eventos ∪ AQ1KM)',
        'Índice de Jaccard (IoU)',
    ],
    'Valor': [
        f'{area_intersec_km2:,.2f} km²',
        f'{area_dif_evento_km2:,.2f} km²',
        f'{area_dif_aq1km_km2:,.2f} km²',
        f'{area_uniao_km2:,.2f} km²',
        f'{iou:.4f}',
    ],
}))

## 5. Visualização espacial

Três cartogramas sintetizam o resultado: feições isoladas por conjunto e feições pareadas em camada combinada.

In [ ]:
# Mapa 1 — Detecções AQ1KM isoladas (sem correspondência em Eventos)
if n_aq1km_isol == 0:
    print('Nenhuma detecção AQ1KM isolada.')
else:
    display(renderizar_mapa(gdf_brasil, [
        (gdf_aq1km_isol, 'AQ1KM isoladas', STYLE_AQ1KM_ISOL),
    ]))

In [ ]:
# Mapa 2 — Eventos isolados (sem correspondência em AQ1KM)
if n_eventos_isol == 0:
    print('Nenhum evento isolado.')
else:
    display(renderizar_mapa(gdf_brasil, [
        (gdf_eventos_isol, 'Eventos isolados', STYLE_EVENTO_ISOL),
    ]))

In [ ]:
# Mapa 3 — Feições pareadas (correspondência confirmada)
display(renderizar_mapa(gdf_brasil, [
    (gdf_aq1km_par,   'AQ1KM pareadas',   STYLE_AQ1KM_PAR),
    (gdf_eventos_par, 'Eventos pareados', STYLE_EVENTO_PAR),
]))